In [1]:
!pip install transformers[torch] pandas scikit-learn

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

WORKSPACE_DIR = '/content/drive/MyDrive/IE403/sentiment-analysis-game-domain/notebook'
os.chdir(WORKSPACE_DIR)

# Khai báo các đường dẫn tương đối từ thư mục notebook
DATA_DIR = '../data/split'
MODEL_DIR = '../models'
PRED_DIR = '../data/pred'

# Tạo thư mục nếu chưa tồn tại
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Tên của 10 cột nhãn khía cạnh
ASPECT_LABELS = [
    'graphics', 'matchmaking', 'store & microtransactions', 'technical_issue',
    'mechanics', 'developer_support', 'event', 'community', 'hero_design', 'difficulty'
]

class GameReviewDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_len):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        review = str(row['review'])

        # Tokenize
        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        # Gom 10 nhãn khía cạnh thành 1 tensor duy nhất
        labels = row[ASPECT_LABELS].to_list()

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# Khởi tạo Tokenizer cho XLM-RoBERTa
PRE_TRAINED_MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
class MultiAspectSentimentModel(nn.Module):
    def __init__(self, model_name, n_aspects=10, n_classes=3):
        super(MultiAspectSentimentModel, self).__init__()
        self.n_aspects = n_aspects
        self.n_classes = n_classes

        self.roberta = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(p=0.3)
        # 10 khía cạnh * 3 nhãn phân lớp = 30 logits đầu ra
        self.out = nn.Linear(self.roberta.config.hidden_size, n_aspects * n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[0][:, 0, :] # Lấy token đại diện câu <s>
        output = self.drop(pooled_output)
        logits = self.out(output)

        # Biến đổi hình dạng tensor về: [batch_size, 10, 3]
        return logits.view(-1, self.n_aspects, self.n_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiAspectSentimentModel(PRE_TRAINED_MODEL_NAME)
model = model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 10

# Thiết lập bộ nạp dữ liệu DataLoader
train_dataset = GameReviewDataset(os.path.join(DATA_DIR, 'train.csv'), tokenizer, MAX_LEN)
val_dataset = GameReviewDataset(os.path.join(DATA_DIR, 'val.csv'), tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Thiết lập bộ tối ưu hóa Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=1e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

loss_fn = nn.CrossEntropyLoss().to(device)

# 1. Hàm huấn luyện (Train Epoch)
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        loss = 0
        for i in range(10):
            loss += loss_fn(outputs[:, i, :], labels[:, i])

        losses.append(loss.item())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return np.mean(losses)

# 2. Hàm đánh giá (Eval Epoch) - Viết tương tự train_epoch nhưng loại bỏ backpropagation
def eval_epoch(model, data_loader, loss_fn, device, n_examples):
    model = model.eval() # Chuyển mô hình sang chế độ evaluation (tắt dropout)
    losses = []

    with torch.no_grad(): # Tắt tính toán gradient để giải phóng bộ nhớ và tăng tốc
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            loss = 0
            for i in range(10):
                loss += loss_fn(outputs[:, i, :], labels[:, i])

            losses.append(loss.item())

    return np.mean(losses)

In [13]:
print("Bắt đầu quá trình huấn luyện...")

best_loss = float('inf')

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    # Huấn luyện mô hình và thu về train loss
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler, len(train_dataset))
    print(f'Train loss: {train_loss:.4f}')

    # Thực hiện đánh giá trên tập Validation thông qua hàm eval_epoch vừa viết
    val_loss = eval_epoch(model, val_loader, loss_fn, device, len(val_dataset))
    print(f'Val loss:   {val_loss:.4f}')

    # Lưu model: Sử dụng trực tiếp dữ liệu val_loss làm điều kiện so sánh để lưu lại trạng thái tốt nhất
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_xlm_roberta_model.bin'))
        print("Đã lưu best model dựa trên kết quả Val Loss!")

Bắt đầu quá trình huấn luyện...
Epoch 1/10
----------
Train loss: 1.3038
Val loss:   1.8227
Đã lưu best model dựa trên kết quả Val Loss!
Epoch 2/10
----------
Train loss: 1.1883
Val loss:   1.7986
Đã lưu best model dựa trên kết quả Val Loss!
Epoch 3/10
----------
Train loss: 1.0819
Val loss:   1.8413
Epoch 4/10
----------
Train loss: 0.9893
Val loss:   1.8285
Epoch 5/10
----------
Train loss: 0.9177
Val loss:   1.8893
Epoch 6/10
----------
Train loss: 0.8591
Val loss:   1.8862
Epoch 7/10
----------
Train loss: 0.8067
Val loss:   1.9250
Epoch 8/10
----------
Train loss: 0.7649
Val loss:   1.9082
Epoch 9/10
----------
Train loss: 0.7272
Val loss:   1.9251
Epoch 10/10
----------
Train loss: 0.7079
Val loss:   1.9239


In [14]:
def get_predictions(model, data_loader):
    model = model.eval()
    review_texts = []
    predictions = []

    with torch.no_grad():
        for d in data_loader:
            texts = d["review_text"]
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Lấy argmax ở chiều cuối cùng (classes) để ra nhãn (0, 1 hoặc 2)
            # outputs shape: [batch_size, 10, 3] -> preds shape: [batch_size, 10]
            _, preds = torch.max(outputs, dim=2)

            review_texts.extend(texts)
            predictions.extend(preds.cpu().numpy())

    return review_texts, np.array(predictions)

# Load test data
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
test_dataset = GameReviewDataset(os.path.join(DATA_DIR, 'test.csv'), tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Load lại best model để dự đoán
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_xlm_roberta_model.bin')))

print("Bắt đầu dự đoán trên tập test...")
test_texts, test_preds = get_predictions(model, test_loader)

# Tạo DataFrame để lưu kết quả
pred_df = pd.DataFrame()
pred_df['id'] = test_df['id'] # Lấy ID từ test gốc
pred_df['review'] = test_texts

# Gán 10 cột dự đoán vào DataFrame
for idx, aspect in enumerate(ASPECT_LABELS):
    pred_df[aspect] = test_preds[:, idx]

# Lưu ra thư mục data/pred/
output_path = os.path.join(PRED_DIR, 'test_predictions.csv')
pred_df.to_csv(output_path, index=False)
print(f"Hoàn tất! Kết quả dự đoán đã được lưu tại: {output_path}")

Bắt đầu dự đoán trên tập test...
Hoàn tất! Kết quả dự đoán đã được lưu tại: ../data/pred/test_predictions.csv


In [15]:
# 8. ĐÁNH GIÁ CHỈ SỐ METRICS TRÊN TẬP TEST
from sklearn.metrics import accuracy_score, f1_score

print("BÁO CÁO HIỆU NĂNG TRÊN TẬP KIỂM THỬ (TEST SET)")
print("=" * 60)
print(f"{'Khía cạnh (Aspect)':<30} | {'Accuracy':<10} | {'F1-Macro':<10}")
print('-' * 60)

# Lấy nhãn thực tế (Ground Truth) từ test_df
# Đảm bảo test.csv của bạn có chứa 10 cột nhãn này
y_true = test_df[ASPECT_LABELS].values
y_pred = test_preds

total_f1_test = 0

# Tính toán và in ra cho từng khía cạnh
for idx, aspect in enumerate(ASPECT_LABELS):
    # Lấy nhãn thật và nhãn dự đoán của cột hiện tại
    true_labels = y_true[:, idx]
    pred_labels = y_pred[:, idx]

    # Tính metrics
    acc = accuracy_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels, average='macro', zero_division=0)

    total_f1_test += f1

    # In ra từng dòng
    print(f"{aspect:<30} | {acc:.4f}     | {f1:.4f}")

print('-' * 60)

# In ra điểm F1-Macro trung bình của tất cả 10 khía cạnh
avg_f1_test = total_f1_test / len(ASPECT_LABELS)
print(f"{'TRUNG BÌNH TỔNG (AVERAGE)':<30} | {'-':<10} | {avg_f1_test:.4f}")
print("=" * 60)

BÁO CÁO HIỆU NĂNG TRÊN TẬP KIỂM THỬ (TEST SET)
Khía cạnh (Aspect)             | Accuracy   | F1-Macro  
------------------------------------------------------------
graphics                       | 0.9692     | 0.4193
matchmaking                    | 0.8786     | 0.5723
store & microtransactions      | 0.9711     | 0.5219
technical_issue                | 0.9210     | 0.6144
mechanics                      | 0.8998     | 0.5814
developer_support              | 0.8863     | 0.7693
event                          | 0.9807     | 0.4422
community                      | 0.9461     | 0.6242
hero_design                    | 0.9711     | 0.4537
difficulty                     | 0.9711     | 0.3701
------------------------------------------------------------
TRUNG BÌNH TỔNG (AVERAGE)      | -          | 0.5369
